# Notebook 5 — Feature Engineering & Preprocessing

This notebook builds the final machine-learning features based on the findings from the EDA notebook.

The main goals are to:

- Create useful temporal, geographic, order-value, item-count, and payment features.
- Remove identifiers and features that are not available at prediction time.
- Handle missing values.
- Encode categorical variables.
- Apply transformations to numerical features when appropriate.
- Fit preprocessing steps only on the training split.
- Apply the fitted transformations to validation and test splits.
- Save the fitted preprocessing pipeline and transformed datasets for the modeling stage.

To avoid data leakage, the prediction point is assumed to be the order purchase time. Therefore, features describing events that happen after the order is placed are not used.

In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib

# Load the three data splits
train = pd.read_csv("../artifacts/train.csv")
val = pd.read_csv("../artifacts/validation.csv")
test = pd.read_csv("../artifacts/test.csv")

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Test shape:", test.shape)

Train shape: (67533, 21)
Validation shape: (14471, 21)
Test shape: (14472, 21)


## Step 2 — Date Preparation

The purchase timestamp is converted to datetime so that temporal features can be extracted from it.

Only information available at the time of purchase will be used for feature engineering.

In [4]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for df in [train, val, test]:
    for column in date_columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")

print(train["order_purchase_timestamp"].dtype)

datetime64[ns]


## Step 3 — Remove Leakage and Identifier Columns

Because the prediction is made at order purchase time, columns describing future delivery events must not be used.

The following types of columns are excluded:

- Order identifiers.
- Delivery timestamps.
- Estimated delivery dates.
- Order status, because it can reveal information about the order after creation.
- Other fields that directly describe future delivery outcomes.

Customer city is also excluded because the EDA showed very high cardinality and many rare categories. Customer state is retained because it has only 27 categories and showed differences in late-delivery rates.

In [5]:
## Step 3 — Remove Leakage and Identifier Columns

excluded_columns = [
    # Identifiers
    "order_id",
    "customer_id",
    "customer_unique_id",
    # Target
    "late",
    # Potential leakage / future information
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    # High-cardinality categorical feature
    "customer_city",
]

print("Excluded columns:")
for column in excluded_columns:
    print("-", column)

Excluded columns:
- order_id
- customer_id
- customer_unique_id
- late
- order_status
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date
- customer_city


## Step 4 — Create Engineered Features

Based on the EDA findings, the following feature groups are created:

### Temporal features
- Purchase year
- Purchase month
- Purchase weekday
- Purchase hour
- Weekend indicator
- Holiday indicator

### Geographic features
- Customer state
- Customer ZIP-code prefix

### Order-value features
- Total price
- Total freight
- Price per item
- Freight per item
- Freight-to-price ratio

### Order-composition features
- Item count
- Number of unique products
- Number of unique sellers

### Payment features
- Total payment amount
- Number of payments
- Payment amount per item

Log-transformed versions of highly skewed monetary and count variables are also created to reduce the impact of extreme values while retaining the original observations.

In [6]:
def create_features(df):
    df = df.copy()

    # --------------------------------------------------
    # Temporal features
    # --------------------------------------------------
    purchase_time = df["order_purchase_timestamp"]

    df["purchase_year"] = purchase_time.dt.year
    df["purchase_month"] = purchase_time.dt.month
    df["purchase_day"] = purchase_time.dt.day
    df["purchase_weekday"] = purchase_time.dt.weekday
    df["purchase_hour"] = purchase_time.dt.hour

    # Weekend indicator
    df["is_weekend"] = (purchase_time.dt.weekday >= 5).astype(int)

    # Brazilian holiday indicator
    try:
        import holidays

        years = purchase_time.dt.year.dropna().astype(int).unique()
        brazil_holidays = holidays.Brazil(years=years)

        df["is_holiday"] = purchase_time.dt.date.isin(brazil_holidays).astype(int)

    except ImportError:
        print("holidays package not installed. Using 0 for is_holiday.")
        df["is_holiday"] = 0

    # --------------------------------------------------
    # Order-value features
    # --------------------------------------------------

    # Avoid division by zero
    item_count_safe = df["item_count"].replace(0, np.nan)

    df["price_per_item"] = df["total_price"] / item_count_safe

    df["freight_per_item"] = df["total_freight"] / item_count_safe

    df["freight_price_ratio"] = df["total_freight"] / df["total_price"].replace(
        0, np.nan
    )

    # --------------------------------------------------
    # Payment features
    # --------------------------------------------------

    df["payment_per_item"] = df["payment_total"] / item_count_safe

    # --------------------------------------------------
    # Log transformations for skewed features
    # --------------------------------------------------

    log_columns = [
        "total_price",
        "total_freight",
        "payment_total",
        "item_count",
        "unique_products",
        "unique_sellers",
        "payment_count",
    ]

    for column in log_columns:
        df[f"log_{column}"] = np.log1p(df[column].clip(lower=0))

    # --------------------------------------------------
    # Remove columns that should not enter the model
    # --------------------------------------------------

    df = df.drop(
        columns=[col for col in excluded_columns if col in df.columns], errors="ignore"
    )

    # Remove raw timestamp after extracting features
    df = df.drop(columns=["order_purchase_timestamp"], errors="ignore")

    return df

In [7]:
## Step 5 — Apply Feature Engineering

X_train = create_features(train)
X_val = create_features(val)
X_test = create_features(test)

y_train = train["late"].copy()
y_val = val["late"].copy()
y_test = test["late"].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nTarget distribution:")
print(y_train.value_counts())

X_train: (67533, 28)
X_val: (14471, 28)
X_test: (14472, 28)

Target distribution:
late
0    61436
1     6097
Name: count, dtype: int64


In [8]:
X_train = create_features(train)

print(X_train["is_holiday"].value_counts(dropna=False))

is_holiday
0    66462
1     1071
Name: count, dtype: int64


## Step 6 — Review Engineered Features

The engineered datasets are inspected to verify that:

- Expected features were created.
- Leakage columns are not present.
- The target variable is separated from the input features.
- Train, validation, and test datasets contain the same feature structure before preprocessing.

In [49]:
print("Training features:")
print(X_train.columns.tolist())

print("\nNumber of training features:", X_train.shape[1])

print("\nLeakage check:")

leakage_columns = [
    "late",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
]

remaining_leakage = [column for column in leakage_columns if column in X_train.columns]

print("Remaining leakage columns:", remaining_leakage)

Training features:
['customer_zip_code_prefix', 'customer_state', 'item_count', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'payment_total', 'payment_count', 'order_year', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour', 'is_weekend', 'is_holiday', 'price_per_item', 'freight_per_item', 'freight_price_ratio', 'payment_per_item', 'log_total_price', 'log_total_freight', 'log_payment_total', 'log_item_count', 'log_unique_products', 'log_unique_sellers', 'log_payment_count']

Number of training features: 28

Leakage check:
Remaining leakage columns: []


## Step 7 — Identify Feature Types

Numerical features will be imputed using the median and standardized.

Categorical features will be imputed using the most frequent category and encoded using one-hot encoding.

`customer_state` is retained as the main geographic categorical feature because the EDA showed that it has relatively low cardinality and meaningful differences in late-delivery rates.

In [50]:
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

numerical_features = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

Categorical features:
['customer_state']

Numerical features:
['customer_zip_code_prefix', 'item_count', 'total_price', 'total_freight', 'unique_products', 'unique_sellers', 'payment_total', 'payment_count', 'order_year', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour', 'is_weekend', 'is_holiday', 'price_per_item', 'freight_per_item', 'freight_price_ratio', 'payment_per_item', 'log_total_price', 'log_total_freight', 'log_payment_total', 'log_item_count', 'log_unique_products', 'log_unique_sellers', 'log_payment_count']

Number of categorical features: 1
Number of numerical features: 27


## Step 8 — Build the Preprocessing Pipeline

A single preprocessing pipeline is used to ensure that the same transformations are applied consistently.

For numerical features:

1. Missing values are replaced using the median.
2. Features are standardized using `StandardScaler`.

For categorical features:

1. Missing values are replaced using the most frequent category.
2. Categories are converted using one-hot encoding.
3. Unknown categories in validation or test data are ignored instead of causing errors.

The preprocessing pipeline will be fitted only on the training data to prevent data leakage.

In [51]:
numerical_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


## Step 9 — Fit Preprocessing on Training Data

The preprocessing pipeline is fitted only on the training split.

The validation and test sets are transformed using the already-fitted pipeline.

This prevents information from the validation or test sets from influencing imputation, scaling, or category encoding.

In [52]:
# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and test using the fitted pipeline
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (67533, 54)
Processed validation shape: (14471, 54)
Processed test shape: (14472, 54)


## Step 10 — Extract Processed Feature Names

After preprocessing, the numerical features and one-hot encoded state features are combined into the final feature matrix.

The feature names are extracted from the fitted preprocessing pipeline so they can be saved and used later during model training and inference.

In [53]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nFirst 20 features:")
print(feature_names[:20])

Number of processed features: 54

First 20 features:
['numerical__customer_zip_code_prefix' 'numerical__item_count'
 'numerical__total_price' 'numerical__total_freight'
 'numerical__unique_products' 'numerical__unique_sellers'
 'numerical__payment_total' 'numerical__payment_count'
 'numerical__order_year' 'numerical__purchase_year'
 'numerical__purchase_month' 'numerical__purchase_day'
 'numerical__purchase_weekday' 'numerical__purchase_hour'
 'numerical__is_weekend' 'numerical__is_holiday'
 'numerical__price_per_item' 'numerical__freight_per_item'
 'numerical__freight_price_ratio' 'numerical__payment_per_item']


## Step 11 — Verify the Processed Data

The processed datasets are checked to ensure that preprocessing removed missing values and produced a consistent feature structure across all splits.

In [54]:
print("Train missing values:", np.isnan(X_train_processed).sum())
print("Validation missing values:", np.isnan(X_val_processed).sum())
print("Test missing values:", np.isnan(X_test_processed).sum())

print("\nShapes:")
print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

Train missing values: 0
Validation missing values: 0
Test missing values: 0

Shapes:
Train: (67533, 54)
Validation: (14471, 54)
Test: (14472, 54)


## Step 12 — Save the Preprocessing Pipeline

The fitted preprocessing pipeline is saved so that the exact same transformations can be reused during model training and future inference.

In [55]:
os.makedirs("../artifacts", exist_ok=True)

joblib.dump(preprocessor, "../artifacts/preprocessor.joblib")

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


## Step 13 — Save Processed Datasets

The processed train, validation, and test feature matrices are saved as NumPy arrays for use during model training.

The target arrays are also saved separately to keep the features and labels clearly separated.

In [56]:
import numpy as np
import os

# Save processed feature matrices
np.save("../artifacts/X_train_processed.npy", X_train_processed)

np.save("../artifacts/X_val_processed.npy", X_val_processed)

np.save("../artifacts/X_test_processed.npy", X_test_processed)

# Save target arrays
np.save("../artifacts/y_train.npy", y_train.to_numpy())

np.save("../artifacts/y_val.npy", y_val.to_numpy())

np.save("../artifacts/y_test.npy", y_test.to_numpy())

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


## Step 14 — Save Feature Names

The final processed feature names are saved to ensure that the meaning and order of the model inputs can be tracked during the modeling stage.

In [59]:
pd.Series(feature_names).to_csv(
    "../artifacts/feature_names.csv", index=False, header=["feature_name"]
)

print("Feature names saved successfully.")

Feature names saved successfully.


## Step 15 — Final Verification

The final artifacts are verified to confirm that the preprocessing pipeline, processed datasets, target arrays, and feature names were saved successfully.

In [58]:
print("Final verification")
print("------------------")

print("Train:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

print("\nFeatures:", len(feature_names))

print("\nSaved artifacts:")
for filename in [
    "preprocessor.joblib",
    "X_train_processed.npy",
    "X_val_processed.npy",
    "X_test_processed.npy",
    "y_train.npy",
    "y_val.npy",
    "y_test.npy",
    "feature_names.csv",
]:
    path = f"../artifacts/{filename}"
    print(filename, "✓" if os.path.exists(path) else "✗")

Final verification
------------------
Train: (67533, 54)
Validation: (14471, 54)
Test: (14472, 54)

Features: 54

Saved artifacts:
preprocessor.joblib ✓
X_train_processed.npy ✓
X_val_processed.npy ✓
X_test_processed.npy ✓
y_train.npy ✓
y_val.npy ✓
y_test.npy ✓
feature_names.csv ✓


## Findings

The feature engineering and preprocessing stage produced a consistent set of 54 machine-learning features.

The main transformations included:

- Temporal features extracted from the order purchase timestamp, including year, month, day, weekday, hour, weekend indicator, and Brazilian holiday indicator.

- Order-value and item-level features such as price per item, freight per item, freight-to-price ratio, and payment per item.

- Log transformations applied to highly right-skewed numerical variables.

- High-cardinality identifiers such as `customer_unique_id` and `customer_city` were excluded from the model inputs.

- Delivery-related timestamps and other future information were excluded to prevent data leakage.

- A final leakage check confirmed that target and future-delivery columns were not present in the model input features.

- `customer_state` was retained as a categorical feature and encoded using one-hot encoding.

- Missing numerical values were handled using median imputation, while missing categorical values were handled using the most frequent category.

- Numerical features were standardized.

- The preprocessing pipeline was fitted only on the training data and then applied to the validation and test sets.

The final processed datasets contain 54 features with consistent dimensions across all three splits:

- Training: 67,533 samples × 54 features
- Validation: 14,471 samples × 54 features
- Test: 14,472 samples × 54 features

The fitted preprocessing pipeline and processed datasets were saved as artifacts for use in the modeling stage.